## Notebook 08 — SHAP Analysis on LightGBM Model

---

This notebook:
1. Loads + merges data, drops >80% missing columns, removes `TransactionID`
2. Performs stratified 80/20 train-validation split
3. Converts object columns to 'category' dtype
4. Trains the same baseline LightGBM classifier (from notebook 07)
5. Performs SHAP analysis:
   - Global explainability (summary plots)
   - Local explainability (force/waterfall plots)
   - Top 15 features by mean absolute SHAP value

> **Uses TreeExplainer on 10,000 validation samples.**

---
## 📦 Step 0 — Import Libraries, Load & Split Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../Datasets/'
TARGET    = 'isFraud'

# Load & merge
print('Loading and merging...')
train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')
train_raw = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity

# Drop >80% missing columns
missing_pct  = train_raw.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
train = train_raw.drop(columns=cols_to_drop)
del train_raw

# Drop identifier column
train = train.drop(columns=['TransactionID'], errors='ignore')

# Stratified 80/20 split
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
del train, X, y

print(f'✅ X_train shape : {X_train.shape}')
print(f'✅ X_val shape   : {X_val.shape}')
print(f'✅ Train fraud % : {y_train.mean()*100:.2f}%')
print(f'✅ Val fraud %   : {y_val.mean()*100:.2f}%')

---
## 🔄 Step 1 — Convert Object Columns to Category Dtype

In [ ]:
# Identify object columns
object_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f'Converting {len(object_cols)} object columns to category dtype...')

# Convert to category dtype in both train and validation sets
for col in object_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

print(f'✅ Converted {len(object_cols)} categorical columns')

---
## 🚀 Step 2 — Train LightGBM Model

In [ ]:
# Initialize and train the same LightGBM model from notebook 07
print('Training LightGBM model...')

model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    scale_pos_weight=27.6,
    random_state=42,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=False)
    ]
)

print(f'✅ Model trained (Best iteration: {model.best_iteration_})')

---
## 🎲 Step 3 — Prepare Sample for SHAP Analysis

In [ ]:
# Sample 10,000 validation rows to avoid memory issues
np.random.seed(42)
sample_size = min(10000, len(X_val))
sample_indices = np.random.choice(X_val.index, size=sample_size, replace=False)
X_val_sample = X_val.loc[sample_indices]
y_val_sample = y_val.loc[sample_indices]

print(f'✅ Sampled {sample_size:,} validation rows for SHAP analysis')
print(f'   Fraud rate in sample: {y_val_sample.mean()*100:.2f}%')

---
## 🔍 Step 4 — Compute SHAP Values

In [ ]:
# Initialize SHAP TreeExplainer
print('Initializing SHAP TreeExplainer...')
explainer = shap.TreeExplainer(model)

# Compute SHAP values
print('Computing SHAP values (this may take a few minutes)...')
shap_values = explainer(X_val_sample)

print('✅ SHAP values computed')
print(f'   Shape: {shap_values.values.shape}')

---
## 📊 Step 5A — Global Explainability: Summary Plots

In [ ]:
# SHAP Summary Plot - Bar (Feature Importance by Mean Absolute SHAP)
print('Generating SHAP summary plot (bar)...')
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_val_sample, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Feature Importance (Mean Absolute SHAP Value)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Bar plot generated')

In [ ]:
# SHAP Summary Plot - Dot (Feature Impact with Value Distribution)
print('Generating SHAP summary plot (dot)...')
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_val_sample, max_display=20, show=False)
plt.title('SHAP Summary Plot (Feature Impact)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Dot plot generated')

---
## 🎯 Step 5B — Top 15 Features by Mean Absolute SHAP Value

In [ ]:
# Compute mean absolute SHAP values for each feature
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

# Create DataFrame
feature_importance_shap = pd.DataFrame({
    'Feature': X_val_sample.columns,
    'Mean Absolute SHAP': mean_abs_shap
}).sort_values('Mean Absolute SHAP', ascending=False).reset_index(drop=True)

feature_importance_shap.index += 1
top15 = feature_importance_shap.head(15)

# Print top 15 features
W = 65
sep = '=' * W
dash = '-' * W
hdr = f'{"Rank":<6} {"Feature":<30} {"Mean |SHAP|":>20}'

print(sep)
print('   TOP 15 FEATURES — By Mean Absolute SHAP Value')
print(sep)
print(hdr)
print(dash)
for rank, row in top15.iterrows():
    print(
        f'{rank:<6} {row["Feature"]:<30} '
        f'{row["Mean Absolute SHAP"]:>20.6f}'
    )
print(sep)

---
## 🔬 Step 6 — Local Explainability: Individual Predictions

In [ ]:
# Find one fraud and one non-fraud transaction in the sample
fraud_idx = y_val_sample[y_val_sample == 1].index[0]
non_fraud_idx = y_val_sample[y_val_sample == 0].index[0]

# Get their positions in the sample
fraud_pos = list(X_val_sample.index).index(fraud_idx)
non_fraud_pos = list(X_val_sample.index).index(non_fraud_idx)

print(f'Selected transactions:')
print(f'  Fraud transaction (label=1): index {fraud_idx}, position {fraud_pos}')
print(f'  Non-fraud transaction (label=0): index {non_fraud_idx}, position {non_fraud_pos}')

### Local Explainability — Fraud Transaction (label=1)

In [ ]:
# SHAP Waterfall Plot for Fraud Transaction
print('Generating waterfall plot for FRAUD transaction...')
plt.figure(figsize=(10, 8))
shap.waterfall_plot(shap_values[fraud_pos], max_display=15, show=False)
plt.title(f'SHAP Waterfall Plot: Fraud Transaction (Index {fraud_idx})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Fraud transaction waterfall plot generated')

### Local Explainability — Non-Fraud Transaction (label=0)

In [ ]:
# SHAP Waterfall Plot for Non-Fraud Transaction
print('Generating waterfall plot for NON-FRAUD transaction...')
plt.figure(figsize=(10, 8))
shap.waterfall_plot(shap_values[non_fraud_pos], max_display=15, show=False)
plt.title(f'SHAP Waterfall Plot: Non-Fraud Transaction (Index {non_fraud_idx})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Non-fraud transaction waterfall plot generated')

---
## 📝 Summary

SHAP analysis complete! 

**Global Insights:**
- Summary plots show which features have the highest overall impact
- Bar plot ranks features by mean absolute SHAP value
- Dot plot shows how feature values affect predictions (red=high, blue=low)

**Local Insights:**
- Waterfall plots explain individual predictions
- Shows how each feature pushed the prediction up or down
- Base value (average model output) → final prediction